# Modelo de Otimização de Orçamento

**Projeto:** Social Wave — Otimização de Campanhas de Marketing  
**Fase:** Recomendação (Fase 3)  
**Objetivo:** Encontrar a alocação de orçamento que minimiza o CPA global

---

## Perguntas que este notebook responde

| # | Pergunta | O que descobrimos | Seção |
|---|----------|-------------------|-------|
| **3.1** | Qual a **alocação ótima** de orçamento entre canais? | Distribuição ideal para minimizar CPA | 3.1 |
| **3.2** | Qual o **CPA projetado** com a alocação ótima vs. atual? | Ganho de eficiência | 3.2 |
| **3.3** | Qual o **risco** de realocar verba (perda de volume)? | Trade-off custo vs. volume | 3.3 |

---

## Dados de Entrada
Base processada (`campanhas_base_processada.pkl`)  
Resumo por canal (`resumo_por_canal.pkl`)

---

## Entregáveis
- Alocação ótima de orçamento por canal (%)
- CPA projetado: atual vs. otimizado
- Análise de trade-off: custo vs. volume

## Setup e Carregamento

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURAÇÕES DE VISUAIS
# ============================================================
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

np.random.seed(42)

# ============================================================
# CARREGAR DADOS
# ============================================================

PICKLE_PATH = r'C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl'
RESUMO_PATH = r'C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\resumo_por_canal.pkl'

try:
    df = pd.read_pickle(PICKLE_PATH)
    resumo = pd.read_pickle(RESUMO_PATH)
    
    print('=' * 60)
    print('📊 DADOS CARREGADOS')
    print('=' * 60)
    print(f'Base: {PICKLE_PATH}')
    print(f'   Dimensoes: {df.shape[0]:,} x {df.shape[1]}')
    print(f'Resumo: {RESUMO_PATH}')
    print(f'   Canais: {len(resumo)}')
    
    print(f'\n📊 Resumo por canal:')
    print(resumo[['Gasto_Total', 'Conversao_Total', 'CPA']].to_string())
    
except FileNotFoundError as e:
    print(f'❌ Arquivo nao encontrado: {e}')
    print('   Execute os Notebooks 01 e 02 primeiro.')
    raise

📊 DADOS CARREGADOS
Base: C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\campanhas_base_processada.pkl
   Dimensoes: 23,315 x 9
Resumo: C:\Users\ricar\Documents\projetos\social-wave-campaign-optimization\data\resumo_por_canal.pkl
   Canais: 6

📊 Resumo por canal:
               Gasto_Total  Conversao_Total   CPA
Canal_Nome                                       
Meta Ads        6040563.96          1556536  3.88
Twitter Ads    12624414.68          2598483  4.86
TikTok          5622544.31           366198 15.35
YouTube Ads    14523324.07           533457 27.22
Google Search  26104521.92           640473 40.76
LinkedIn Ads     785726.71            16552 47.47
